# Automated EDA Tools

Exploratory Data Analysis traditionally involves manually writing code to plot distributions, check correlations, identify missing values, and summarise statistics. **Automated EDA tools** compress this from hours of work into a single function call.

## Tools Covered

| Tool | Install | Output | Best for |
|---|---|---|---|
| **ydata-profiling** | `pip install ydata-profiling` | Interactive HTML report | Comprehensive dataset audit |
| **sweetviz** | `pip install sweetviz` | Visual HTML report | Train/test comparison |
| **dtale** | `pip install dtale` | Live web UI | Interactive exploration |
| **dataprep** | `pip install dataprep` | HTML report | Fast, clean profiling |
| **autoviz** | `pip install autoviz` | Auto-generated charts | Quick visual patterns |

## Important Caveat
> Automated tools **accelerate** exploration — they do not replace analytical thinking. Use them to find where to look, then investigate with domain knowledge.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)

# Shared dataset used across all tools
n = 1000
df = pd.DataFrame({
    'age':         np.random.randint(18, 70, n).astype(float),
    'income':      np.random.exponential(50000, n).round(2),
    'score':       np.random.uniform(1, 10, n).round(2),
    'city':        np.random.choice(['Mumbai', 'Delhi', 'Bangalore', 'Chennai'], n),
    'tier':        np.random.choice(['Bronze', 'Silver', 'Gold', 'Platinum'], n),
    'churn':       np.random.choice([0, 1], n, p=[0.8, 0.2]),
    'signup_date': pd.date_range('2021-01-01', periods=n, freq='8h'),
    'revenue':     np.random.exponential(3000, n).round(2),
})

# Add correlated feature and missing values
df['spend']    = df['income'] * np.random.uniform(0.05, 0.3, n)
df.loc[np.random.choice(df.index, 80),  'age']    = np.nan
df.loc[np.random.choice(df.index, 120), 'income'] = np.nan
df.loc[np.random.choice(df.index, 40),  'city']   = np.nan

print(f"Dataset: {df.shape[0]} rows × {df.shape[1]} columns")
df.head()

---
## 0. Manual Baseline — What Auto Tools Automate

Before using any tool, understand what they are doing under the hood. This is the manual version of a profile report.

In [ ]:
def manual_profile(df: pd.DataFrame) -> pd.DataFrame:
    num_df = df.select_dtypes(include='number')
    cat_df = df.select_dtypes(include=['object', 'category'])

    profile = pd.DataFrame(index=df.columns)
    profile['dtype']      = df.dtypes
    profile['null_count'] = df.isnull().sum()
    profile['null_pct']   = (df.isnull().mean() * 100).round(2)
    profile['unique']     = df.nunique()
    profile['unique_pct'] = (df.nunique() / len(df) * 100).round(2)

    for col in num_df.columns:
        profile.loc[col, 'mean']  = num_df[col].mean().round(2)
        profile.loc[col, 'std']   = num_df[col].std().round(2)
        profile.loc[col, 'min']   = num_df[col].min()
        profile.loc[col, 'max']   = num_df[col].max()
        profile.loc[col, 'skew']  = num_df[col].skew().round(3)

    return profile


print("Manual profile:")
manual_profile(df)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from scipy.stats import gaussian_kde

profile = manual_profile(df)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.suptitle('Manual Profile Dashboard — Core Statistics Every Auto EDA Tool Reports',
             fontsize=13, fontweight='bold')

# Panel 1 — Null % per column (sorted descending)
null_pct = profile['null_pct'].fillna(0).sort_values(ascending=False)
null_colors = ['#e63946' if v > 0 else '#2a9d8f' for v in null_pct.values]
axes[0].barh(null_pct.index, null_pct.values,
             color=null_colors, edgecolor='white', alpha=0.85)
axes[0].axvline(5, color='#555', ls='--', lw=1.2, alpha=0.6, label='5% threshold')
axes[0].set_xlabel('Missing %')
axes[0].set_title('Null % per Column\n(red = has missing data)')
axes[0].legend(fontsize=8)
axes[0].spines[['top', 'right']].set_visible(False)

# Panel 2 — Skewness for numeric columns
skew_vals = profile['skew'].dropna().sort_values()
skew_colors = ['#e63946' if abs(v) > 1 else '#2a9d8f' for v in skew_vals.values]
axes[1].barh(skew_vals.index, skew_vals.values, color=skew_colors,
             edgecolor='white', alpha=0.85)
axes[1].axvline( 1, color='#555', ls='--', lw=1.2, alpha=0.6, label='|skew| = 1')
axes[1].axvline(-1, color='#555', ls='--', lw=1.2, alpha=0.6)
axes[1].axvline(0,  color='black', lw=0.8, alpha=0.3)
axes[1].set_xlabel('Skewness')
axes[1].set_title('Skewness per Numeric Column\n(|skew| > 1 = log transform candidate)')
axes[1].legend(fontsize=8)
axes[1].spines[['top', 'right']].set_visible(False)

# Panel 3 — Overlapping normalised KDE (shape comparison across numeric columns)
num_cols_kde = df.select_dtypes(include='number').drop(columns=['churn']).columns[:5]
x_range = np.linspace(0, 1, 300)
for col in num_cols_kde:
    vals = df[col].dropna()
    vals_norm = (vals - vals.min()) / (vals.max() - vals.min() + 1e-9)
    kde = gaussian_kde(vals_norm)
    axes[2].plot(x_range, kde(x_range), label=col, lw=1.8)
axes[2].set_xlabel('Normalised value [0, 1]')
axes[2].set_ylabel('Density')
axes[2].set_title('Normalised KDE — Distribution Shape Comparison\n(uniform = well-spread; spike = skewed)')
axes[2].legend(fontsize=7.5)
axes[2].spines[['top', 'right']].set_visible(False)

plt.tight_layout()
plt.show()

In [ ]:
# Manual correlation heatmap
import matplotlib.pyplot as plt
import seaborn as sns

corr = df.select_dtypes(include='number').corr()

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0, ax=ax)
ax.set_title('Correlation Matrix (manual)')
plt.tight_layout()
plt.show()

---
## 1. ydata-profiling (formerly pandas-profiling)

The most comprehensive automated EDA library. Generates a self-contained interactive HTML report.

```
pip install ydata-profiling
```

### What it reports
- Overview: shape, memory, duplicate rows, missing count
- Per-column: distribution histogram, descriptive statistics, outlier flags, high-cardinality warnings
- Interactions: scatter plots between numeric pairs
- Correlations: Pearson, Spearman, Cramér's V (for categorical)
- Missing value heatmap and co-occurrence matrix
- Sample rows

### Pros
- Most thorough out-of-the-box report
- Handles numeric, categorical, datetime, boolean columns
- Configurable depth (`minimal=True` for large datasets)
- Exportable as HTML or JSON

### Cons
- Slow on large datasets (>100k rows) without `minimal=True`
- HTML report can be overwhelming for stakeholders
- High memory usage on wide datasets

In [ ]:
try:
    from ydata_profiling import ProfileReport

    # Full report — thorough but slow on large datasets
    profile_full = ProfileReport(
        df,
        title='Customer Dataset — Full Profile',
        explorative=True,
    )
    profile_full.to_file('/tmp/ydata_full.html')
    print("Full report → /tmp/ydata_full.html")

    # Minimal report — faster, suitable for large datasets
    profile_minimal = ProfileReport(
        df,
        title='Customer Dataset — Minimal Profile',
        minimal=True,
    )
    profile_minimal.to_file('/tmp/ydata_minimal.html')
    print("Minimal report → /tmp/ydata_minimal.html")

    # Render inline in notebook
    profile_minimal.to_notebook_iframe()

except ImportError:
    print("Install: pip install ydata-profiling")

In [ ]:
# Access programmatic summary without generating HTML
try:
    from ydata_profiling import ProfileReport

    profile = ProfileReport(df, minimal=True)
    desc = profile.get_description()

    print(f"Rows:          {desc.table['n']}")
    print(f"Columns:       {desc.table['n_var']}")
    print(f"Missing cells: {desc.table['n_cells_missing']}")
    print(f"Duplicates:    {desc.table['n_duplicates']}")

except (ImportError, Exception) as e:
    print(f"Note: {e}")

---
## 2. sweetviz

Focused on **comparison**: analysing a single dataset or comparing two (e.g. train vs test, before vs after cleaning).

```
pip install sweetviz
```

### What it reports
- Side-by-side column distributions for two DataFrames
- Association/correlation matrix (numeric and categorical together)
- Target feature analysis — how each column relates to a target variable
- Missing value breakdown

### Pros
- Excellent for train/test distribution comparison (detecting data drift)
- Visual and intuitive — easier for stakeholders than ydata-profiling
- Fast even on medium-sized datasets
- Target analysis built-in (one-click correlation with target)

### Cons
- Less detailed than ydata-profiling per column
- No time-series support
- Cannot export as JSON (HTML only)

In [ ]:
try:
    import sweetviz as sv
    from sklearn.model_selection import train_test_split

    train_df, test_df = train_test_split(df, test_size=0.2, random_state=42)

    # Analyse a single dataset with a target variable
    report_single = sv.analyze(train_df, target_feat='churn')
    report_single.show_html('/tmp/sweetviz_single.html', open_browser=False)
    print("Single dataset report → /tmp/sweetviz_single.html")

    # Compare train vs test — detects distribution drift
    report_compare = sv.compare([train_df, 'Train'], [test_df, 'Test'], target_feat='churn')
    report_compare.show_html('/tmp/sweetviz_compare.html', open_browser=False)
    print("Train vs Test comparison → /tmp/sweetviz_compare.html")

except ImportError:
    print("Install: pip install sweetviz")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from sklearn.model_selection import train_test_split
from scipy.stats import ks_2samp, chi2_contingency

train_df, test_df = train_test_split(df, test_size=0.2, random_state=42)

num_cols_plot = ['age', 'income', 'score', 'revenue', 'spend']

fig, axes = plt.subplots(2, 3, figsize=(15, 7))
fig.suptitle('Train vs Test Distribution — Replicating sweetviz Compare Mode',
             fontsize=13, fontweight='bold')

# Numeric columns: overlapping histograms + KS-test drift flag
for ax, col in zip(axes.flat[:5], num_cols_plot):
    t_vals = train_df[col].dropna()
    te_vals = test_df[col].dropna()
    ax.hist(t_vals,  bins=30, color='#1565C0', alpha=0.5, density=True, label='Train')
    ax.hist(te_vals, bins=30, color='#e63946', alpha=0.5, density=True, label='Test')
    ks_stat, ks_p = ks_2samp(t_vals, te_vals)
    drift  = 'DRIFT' if ks_p < 0.05 else 'OK'
    t_col  = '#e63946' if drift == 'DRIFT' else '#2a9d8f'
    ax.set_title(f'{col}   KS p={ks_p:.3f}  [{drift}]',
                 color=t_col, fontsize=9, fontweight='bold')
    ax.legend(fontsize=7)
    ax.spines[['top', 'right']].set_visible(False)

# Categorical: churn rate comparison (target class balance)
churn_train = train_df['churn'].mean() * 100
churn_test  = test_df['churn'].mean()  * 100
axes[1][2].bar(['Train', 'Test'], [churn_train, churn_test],
               color=['#1565C0', '#e63946'],
               edgecolor='white', linewidth=1.5, alpha=0.85, width=0.4)
for i, val in enumerate([churn_train, churn_test]):
    axes[1][2].text(i, val + 0.3, f'{val:.1f}%',
                    ha='center', fontweight='bold', fontsize=11)
axes[1][2].set_ylabel('Churn rate (%)')
axes[1][2].set_title('Target: Churn Rate\n(should match — class balance check)',
                     fontsize=9)
axes[1][2].spines[['top', 'right']].set_visible(False)

plt.tight_layout()
plt.show()

---
## 3. dtale

Launches an **interactive web UI** for live data exploration — the closest thing to a data browser.

```
pip install dtale
```

### What it provides
- Sortable, filterable spreadsheet view of the DataFrame
- Column-level statistics, charts, and histograms on click
- Correlation matrix and scatter plot builder
- Custom column creation with a GUI
- Code export — generates the Pandas code behind each operation

### Pros
- Most interactive option — ideal for initial exploration and ad hoc queries
- Code export teaches Pandas patterns
- Works in notebooks and standalone browser
- Handles large DataFrames with pagination

### Cons
- Not suitable for automated or scripted reporting
- Requires a running server (not shareable as a file)
- Heavier setup in remote/cloud environments

In [ ]:
try:
    import dtale

    # Launch interactive UI — opens in browser or as iframe in Jupyter
    d = dtale.show(df, subprocess=False, open_browser=False)
    print(f"dtale running at: {d._url}")
    print("Open the URL above in your browser to explore the dataset interactively.")

    # In a Jupyter notebook you can also embed it inline:
    # d.open_browser()  # opens in new browser tab
    # d.notebook()      # embeds as iframe in the notebook

except ImportError:
    print("Install: pip install dtale")

---
## 4. dataprep

Fast profiling with a clean API. The `create_report()` function is particularly quick on large datasets.

```
pip install dataprep
```

### What it reports
- Dataset overview (shape, types, missing values)
- Variable distributions with histogram and box plot
- Correlation analysis
- Missing value visualisation

### Pros
- Fastest profiling library — optimised with Dask under the hood
- Clean, modern HTML output
- Modular: `plot()` for individual column EDA, `plot_correlation()`, `plot_missing()`
- Works well on large datasets

### Cons
- Less detailed than ydata-profiling (no duplicate detection, fewer stat tests)
- Smaller community and fewer configuration options
- No train/test comparison feature

In [ ]:
try:
    from dataprep.eda import create_report, plot, plot_correlation, plot_missing

    # Full report
    report = create_report(df, title='Customer Dataset — DataPrep')
    report.save('/tmp/dataprep_report.html')
    print("Report → /tmp/dataprep_report.html")

    # Modular exploration
    plot(df, 'income')            # distribution of a single column
    plot_correlation(df)          # correlation matrix
    plot_missing(df)              # missing value patterns

except ImportError:
    print("Install: pip install dataprep")

---
## 5. autoviz

Automatically selects and generates the most appropriate chart types for each column and relationship.

```
pip install autoviz
```

### What it generates
- Histograms and KDE plots for numeric columns
- Bar charts for categorical columns
- Scatter plots for numeric pairs
- Box plots split by categorical variables
- Heatmaps for correlation

### Pros
- Zero configuration — just pass the DataFrame
- Automatically picks chart type based on data types
- Can work with CSV files directly
- Lightweight compared to ydata-profiling

### Cons
- Output is static matplotlib/bokeh plots, not an interactive report
- Less control over individual visualisations
- Chart selection can be suboptimal for domain-specific data
- No summary statistics table

In [ ]:
try:
    from autoviz.AutoViz_Class import AutoViz_Class

    AV = AutoViz_Class()

    # target='churn' tells autoviz to focus relationships on the target variable
    dft = AV.AutoViz(
        filename='',          # pass '' to use a DataFrame directly
        sep=',',
        depVar='churn',       # target variable
        dfte=df.select_dtypes(include='number'),  # numeric only for speed
        header=0,
        verbose=1,
        lowess=False,
        chart_format='svg',
        max_rows_analyzed=500,
        max_cols_analyzed=10,
    )

except ImportError:
    print("Install: pip install autoviz")

---
## 6. Tool Comparison

| Criterion | ydata-profiling | sweetviz | dtale | dataprep | autoviz |
|---|---|---|---|---|---|
| **Output format** | HTML report | HTML report | Live web UI | HTML report | Static charts |
| **Speed** | Slow on large data | Fast | Fast | Fastest | Fast |
| **Depth** | Highest | Medium | Medium | Medium | Low |
| **Train/test compare** | Yes (compare mode) | Best-in-class | No | No | No |
| **Interactivity** | Medium | Low | Highest | Low | Low |
| **Shareable file** | Yes | Yes | No | Yes | No |
| **Programmatic access** | Yes (JSON export) | No | Limited | No | No |
| **Large dataset support** | Use minimal=True | Yes | Yes | Best | Yes |
| **Categorical handling** | Yes | Yes | Yes | Yes | Limited |
| **Time-series support** | Yes | No | Yes | No | No |

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np

tools    = ['ydata-profiling', 'sweetviz', 'dtale', 'dataprep', 'autoviz']
criteria = ['Speed', 'Depth', 'Train/Test\nCompare', 'Interactivity',
            'Large\nDataset', 'Categorical\nHandling']

# Scores: 1 = Low, 2 = Medium, 3 = High
scores = np.array([
    [1, 3, 2, 1, 1, 3],   # ydata-profiling
    [2, 2, 3, 1, 2, 3],   # sweetviz
    [2, 2, 1, 3, 2, 3],   # dtale
    [3, 2, 1, 1, 3, 3],   # dataprep
    [2, 1, 1, 1, 2, 1],   # autoviz
])

score_df = pd.DataFrame(scores, index=tools, columns=criteria)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Auto EDA Tool Capabilities — Visual Comparison',
             fontsize=13, fontweight='bold')

# Panel 1 — Heatmap of capability scores
sns.heatmap(score_df, annot=True, fmt='d', cmap='YlGn',
            linewidths=0.6, ax=axes[0], vmin=1, vmax=3,
            cbar_kws={'label': '1=Low  2=Med  3=High'},
            annot_kws={'fontsize': 12})
axes[0].set_title('Capability Score per Tool\n(1 = Low, 2 = Medium, 3 = High)')
axes[0].set_ylabel('')

# Panel 2 — Ranked total score bar chart
totals = scores.sum(axis=1)
tool_colors = ['#1565C0', '#2E7D32', '#E65100', '#6A1B9A', '#00838F']
bars = axes[1].barh(tools, totals, color=tool_colors,
                    edgecolor='white', linewidth=1.5, alpha=0.88, height=0.5)
for bar, val in zip(bars, totals):
    axes[1].text(val + 0.1,
                 bar.get_y() + bar.get_height() / 2,
                 f'{val}/{len(criteria) * 3}',
                 va='center', fontweight='bold', fontsize=11)
axes[1].set_xlabel('Total capability score')
axes[1].set_xlim(0, len(criteria) * 3 + 3)
axes[1].set_title('Overall Score\n(breadth of capabilities — not a quality ranking)')
axes[1].spines[['top', 'right']].set_visible(False)

plt.tight_layout()
plt.show()

## 7. When to Use Which Tool

```
Starting a new dataset for the first time?
  → ydata-profiling (full audit, most thorough)

Need to share a report with a non-technical stakeholder?
  → sweetviz (clean visuals, easy to read)

Checking if train and test distributions match?
  → sweetviz compare mode

Want to interactively filter, sort, and query the data?
  → dtale

Working with a large dataset (>500k rows)?
  → dataprep (fastest) or ydata-profiling with minimal=True

Just want quick charts without configuration?
  → autoviz
```

In [ ]:
# Benchmark: approximate time for each tool on the sample dataset
import time

timings = {}

try:
    from ydata_profiling import ProfileReport
    start = time.time()
    ProfileReport(df, minimal=True).to_file('/tmp/_bench_ydata.html')
    timings['ydata-profiling (minimal)'] = round(time.time() - start, 2)
except ImportError:
    timings['ydata-profiling'] = 'not installed'

try:
    import sweetviz as sv
    start = time.time()
    sv.analyze(df).show_html('/tmp/_bench_sv.html', open_browser=False)
    timings['sweetviz'] = round(time.time() - start, 2)
except ImportError:
    timings['sweetviz'] = 'not installed'

try:
    from dataprep.eda import create_report
    start = time.time()
    create_report(df).save('/tmp/_bench_dp.html')
    timings['dataprep'] = round(time.time() - start, 2)
except ImportError:
    timings['dataprep'] = 'not installed'

print(f"{'Tool':<35} {'Time (s)':>10}")
print('-' * 47)
for tool, t in timings.items():
    print(f"{tool:<35} {str(t):>10}")

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(9, 4))
fig.suptitle('Auto EDA Tools — Execution Time Benchmark',
             fontsize=13, fontweight='bold')

installed   = {k: v for k, v in timings.items() if isinstance(v, float)}
not_inst    = [k for k, v in timings.items() if not isinstance(v, float)]

if installed:
    sorted_items = sorted(installed.items(), key=lambda x: x[1], reverse=True)
    names  = [k for k, _ in sorted_items]
    t_vals = [v for _, v in sorted_items]
    bar_cols = ['#e63946' if v == max(t_vals) else '#3d5a80' for v in t_vals]

    bars = ax.barh(names, t_vals, color=bar_cols, edgecolor='white',
                   linewidth=1.5, alpha=0.88)
    for bar, val in zip(bars, t_vals):
        ax.text(val + max(t_vals) * 0.01,
                bar.get_y() + bar.get_height() / 2,
                f'{val:.2f}s', va='center', fontsize=10, fontweight='bold')
    ax.set_xlabel('Execution time (seconds) — lower is better')
    ax.set_title(f'Benchmark on {len(df):,}-row dataset  |  red = slowest')
    ax.spines[['top', 'right']].set_visible(False)

    if not_inst:
        note = 'Not installed: ' + ', '.join(not_inst)
        ax.text(0.02, -0.12, note, transform=ax.transAxes,
                fontsize=8.5, color='#777', style='italic')
else:
    ax.text(0.5, 0.5,
            'No tools installed yet.\n\nInstall any tool to see timing:\n'
            'pip install ydata-profiling sweetviz dataprep',
            ha='center', va='center', transform=ax.transAxes, fontsize=11,
            bbox=dict(boxstyle='round', facecolor='#E3F2FD', alpha=0.85))
    ax.axis('off')

plt.tight_layout()
plt.show()

---
## 8. Limitations of Automated EDA

Automated tools are powerful but have real limits:

| Limitation | Why it matters |
|---|---|
| **No domain knowledge** | A tool flags `salary=0` as valid; a human knows it means missing |
| **False correlations** | High correlation between two derived columns is reported without context |
| **Misleading on imbalanced targets** | Class imbalance in `churn` may not be prominently surfaced |
| **Datetime handling** | Most tools treat datetime as categorical unless explicitly configured |
| **Speed on wide data** | 1000+ column datasets can cause timeouts or memory errors |
| **Not a substitute for visualisation** | A histogram hides multi-modal distributions that a KDE reveals |

Always **read the report critically**, not as ground truth.

In [ ]:
# Example: automated tools may miss multi-modal distributions
# Two distinct customer segments mixed in one column
segment_a = np.random.normal(loc=30000, scale=5000, size=500)
segment_b = np.random.normal(loc=80000, scale=8000, size=500)
bimodal_income = np.concatenate([segment_a, segment_b])

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# What a summary stat shows
axes[0].bar(['mean', 'median', 'std'],
            [np.mean(bimodal_income), np.median(bimodal_income), np.std(bimodal_income)],
            color='steelblue')
axes[0].set_title('Summary Statistics — Looks Normal')

# What a histogram reveals
axes[1].hist(bimodal_income, bins=60, color='coral', edgecolor='white')
axes[1].axvline(np.mean(bimodal_income), color='black', linestyle='--', label='mean')
axes[1].set_title('Histogram — Reveals Two Segments')
axes[1].legend()

plt.suptitle('Automated tools report the mean; histograms reveal the truth', y=1.02)
plt.tight_layout()
plt.show()

---
## Key Takeaways

- Use **ydata-profiling** for a thorough first audit of any new dataset; add `minimal=True` for speed.
- Use **sweetviz** when you need to compare train/test splits or detect distribution drift.
- Use **dtale** when you want to interactively browse, filter, and query data live.
- Use **dataprep** when working with large datasets and speed matters.
- Use **autoviz** for zero-effort chart generation when you just need a quick visual pass.
- No tool replaces domain knowledge — treat reports as a starting point, not a conclusion.